# Evaluacion Parcial N 1 - Limpieza y preprocesamiento de datos

Notebook organizado por puntos solicitados en instrucciones.md.

Flujo general: Datos crudos -> Limpieza -> Transformacion -> Reduccion de dimensionalidad -> Dataset preparado.

## 0) Configuracion, carga y funciones auxiliares

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 120)

base_dir = Path.cwd()
if base_dir.name.lower() == "notebooks":
    base_dir = base_dir.parent

ruta_raw = base_dir / "data" / "urgencias_noprocesados_grupo04.csv"
ruta_out = base_dir / "outputs" / "datos_limpios.csv"

df_raw = pd.read_csv(ruta_raw)
df_raw.head()

In [ ]:
def diagnostico(df, nombre="DataFrame"):
    print(f"=== {nombre} ===")
    print("Shape:", df.shape)
    print("\nInfo:")
    df.info()
    print("\nValores faltantes por columna:")
    print(df.isna().sum().sort_values(ascending=False).head(15))
    print("\nFilas duplicadas:", df.duplicated().sum())
    print("\nDescribe numerico:")
    display(df.describe().T)

def reporte_iqr(df, col):
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    mask = (df[col] < li) | (df[col] > ls)
    return {"columna": col, "Q1": q1, "Q3": q3, "IQR": iqr, "lim_inf": li, "lim_sup": ls, "n_outliers": int(mask.sum())}

## 1) Diagnostico adecuado del problema
Exploracion breve: se observan variables de establecimiento, ubicacion, causa, conteos de pacientes por tramo etario y costo de atencion.

In [ ]:
diagnostico(df_raw, "Datos crudos")

## 2) Estructuras utilizadas
- list: para manejar conjuntos de columnas (numericas/categoricas).
- dict: para guardar reportes de metricas y outliers.
- numpy array: resultado de transformaciones y operaciones vectorizadas.
- pandas DataFrame: estructura principal para limpiar, analizar y exportar datos.

In [ ]:
df = df_raw.copy()

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()

resumen_estructuras = {
    "n_columnas_numericas": len(numeric_cols),
    "n_columnas_categoricas": len(categorical_cols),
    "primeras_categoricas": categorical_cols[:5],
}

X_np = df[numeric_cols].to_numpy()
print("Tipo list (numeric_cols):", type(numeric_cols))
print("Tipo dict (resumen_estructuras):", type(resumen_estructuras))
print("Tipo np.array (X_np):", type(X_np), "shape=", X_np.shape)
print("Tipo DataFrame (df):", type(df))
resumen_estructuras

## 3) Carga y exploracion inicial (ANTES)
Se aplican los comandos base solicitados para diagnostico inicial.

In [ ]:
print("df.info()")
df.info()
print("\nFaltantes por columna")
display(df.isna().sum().sort_values(ascending=False))
print("Duplicados:", df.duplicated().sum())
print("\nDescribe:")
display(df.describe().T)

## 4) Deteccion de problemas
Se revisan faltantes, duplicados, formatos de fecha inconsistentes y outliers en variables numericas clave.

In [ ]:
problemas = {}
problemas["faltantes_totales"] = int(df.isna().sum().sum())
problemas["duplicados"] = int(df.duplicated().sum())

fecha_parse = pd.to_datetime(df["FechaAtencionTexto"], errors="coerce", dayfirst=True)
problemas["fechas_no_parseables"] = int(fecha_parse.isna().sum())

cols_outlier = [c for c in ["NumTotal", "CostoAtencionCLP"] if c in df.columns]
reporte_outliers = [reporte_iqr(df, c) for c in cols_outlier]
problemas["outliers"] = reporte_outliers

problemas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
if "NumTotal" in df.columns:
    sns.boxplot(x=df["NumTotal"], ax=axes[0], color="#2a9d8f")
    axes[0].set_title("Outliers en NumTotal")
if "CostoAtencionCLP" in df.columns:
    sns.boxplot(x=df["CostoAtencionCLP"], ax=axes[1], color="#e76f51")
    axes[1].set_title("Outliers en CostoAtencionCLP")
plt.tight_layout()
plt.show()

## 5) Tecnicas de limpieza y preprocesamiento
Decisiones aplicadas:
- Duplicados: se eliminan filas repetidas exactas.
- Faltantes numericos: imputacion por mediana (robusta a valores extremos).
- Faltantes categoricos: imputacion por moda.
- Ruido de texto: unificacion de mayusculas/minusculas y espacios.
- Fechas: normalizacion a formato datetime.
- Outliers: recorte por IQR solo en CostoAtencionCLP (winsorizacion simple).

In [ ]:
df_clean = df.copy()

# 1) Eliminar duplicados
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

# 2) Limpiar textos
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
    df_clean[col] = df_clean[col].replace({"nan": np.nan})

# 3) Normalizar fecha y extraer componentes
df_clean["FechaAtencion"] = pd.to_datetime(df_clean["FechaAtencionTexto"], errors="coerce", dayfirst=True)

# 4) Imputacion simple
for col in numeric_cols:
    med = df_clean[col].median()
    df_clean[col] = df_clean[col].fillna(med)

for col in categorical_cols:
    moda = df_clean[col].mode(dropna=True)
    if len(moda) > 0:
        df_clean[col] = df_clean[col].fillna(moda.iloc[0])
    else:
        df_clean[col] = df_clean[col].fillna("desconocido")

# 5) Tratamiento de outliers (solo costo)
if "CostoAtencionCLP" in df_clean.columns:
    q1 = df_clean["CostoAtencionCLP"].quantile(0.25)
    q3 = df_clean["CostoAtencionCLP"].quantile(0.75)
    iqr = q3 - q1
    li = q1 - 1.5 * iqr
    ls = q3 + 1.5 * iqr
    df_clean["CostoAtencionCLP"] = df_clean["CostoAtencionCLP"].clip(lower=li, upper=ls)

df_clean.head()

## 6) Verificacion posterior (DESPUES)
Se comparan las mismas metricas para evidenciar el impacto de la limpieza.

In [ ]:
diagnostico(df_clean, "Datos limpios")

comparacion = pd.DataFrame({
    "metrica": ["filas", "columnas", "faltantes_totales", "duplicados"],
    "antes": [
        df.shape[0],
        df.shape[1],
        int(df.isna().sum().sum()),
        int(df.duplicated().sum()),
    ],
    "despues": [
        df_clean.shape[0],
        df_clean.shape[1],
        int(df_clean.isna().sum().sum()),
        int(df_clean.duplicated().sum()),
    ]
})
comparacion

## 7) Feature engineering y agregaciones
Se crean nuevas caracteristicas desde la fecha y se realizan agregaciones para analisis.

In [ ]:
df_clean["anio_fecha"] = df_clean["FechaAtencion"].dt.year
df_clean["mes_fecha"] = df_clean["FechaAtencion"].dt.month
df_clean["dia_semana"] = df_clean["FechaAtencion"].dt.day_name().str.lower()

ag_region = (
    df_clean.groupby("RegionGlosa", dropna=False)[["NumTotal", "CostoAtencionCLP"]]
    .agg({"NumTotal": "sum", "CostoAtencionCLP": "mean"})
    .sort_values("NumTotal", ascending=False)
    .head(10)
)
ag_region

## 8) Pipeline de transformacion (justificacion)
Se usa pipeline para reproducibilidad, orden y para evitar errores al aplicar pasos de transformacion manualmente.

Pasos aplicados en pipeline:
- Numericas: imputacion por mediana + estandarizacion.
- Categoricas: imputacion por moda + one-hot encoding.

In [ ]:
num_cols_pipe = df_clean.select_dtypes(include=["number"]).columns.tolist()
cat_cols_pipe = df_clean.select_dtypes(exclude=["number", "datetime"]).columns.tolist()

num_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocess = ColumnTransformer(transformers=[
    ("num", num_pipe, num_cols_pipe),
    ("cat", cat_pipe, cat_cols_pipe),
], remainder="drop")

X_prepared = preprocess.fit_transform(df_clean)
print("Shape transformado:", X_prepared.shape)
X_prepared[:2, :8]

## 9) Impacto en analisis/modelo (antes vs despues)
- Menos ruido y menos sesgo por datos inconsistentes.
- Menor varianza por estandarizacion y control de outliers extremos.
- Mejor base para entrenar modelos y generar reportes confiables.

## 10) Analisis final y graficos

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top_causas = (
    df_clean.groupby("Causa", dropna=False)["NumTotal"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)
sns.barplot(x=top_causas.values, y=top_causas.index, ax=axes[0], color="#264653")
axes[0].set_title("Top 10 causas por NumTotal")
axes[0].set_xlabel("Total atenciones")
axes[0].set_ylabel("Causa")

if "PrioridadTriage" in df_clean.columns:
    triage_plot = (
        df_clean.groupby("PrioridadTriage", dropna=False)["NumTotal"]
        .sum()
        .sort_values(ascending=False)
    )
    sns.barplot(x=triage_plot.index, y=triage_plot.values, ax=axes[1], color="#2a9d8f")
    axes[1].set_title("Atenciones por prioridad triage")
    axes[1].set_xlabel("Prioridad")
    axes[1].set_ylabel("Total atenciones")

plt.tight_layout()
plt.show()

## 11) Exportacion del dataset limpio

In [ ]:
ruta_out.parent.mkdir(parents=True, exist_ok=True)
df_clean.to_csv(ruta_out, index=False)
print(f"Archivo generado: {ruta_out}")

## 12) Respuestas finales solicitadas
a) Aprendizajes: importancia del diagnostico inicial, limpieza justificada, y validacion antes/despues.

b) Puntos relevantes: deteccion de formatos mixtos de fecha, control de outliers en costo, uso de pipeline para reproducibilidad.

c) Analisis del proyecto: las graficas respaldan causas dominantes y distribucion de triage para apoyar decisiones.

d) Metodologia usada: CRISP-DM simplificada (comprension de datos, preparacion, analisis y comunicacion de resultados).

e) Reproducibilidad: rutas relativas, pasos secuenciales, pipeline explicito y exportacion final del dataset limpio.